# 08 - Debugging PyTorch: Common Errors

## Learning Objectives

- Recognise and fix the most common PyTorch runtime errors
- Diagnose and resolve shape mismatch errors
- Fix device mismatch errors
- Understand in-place operation errors with autograd
- Debug NaN/Inf in loss (exploding gradients, `log(0)`)
- Handle memory errors with reduced batch size and gradient accumulation
- Avoid common gotchas: forgetting `model.eval()`, `no_grad()`, `zero_grad()`
- Use `torch.autograd.detect_anomaly()` for debugging

## Prerequisites

- Notebooks 01-05 of this series
- Familiarity with training loops
- PyTorch installed (`pip install torch`)

## Table of Contents

1. [Shape Mismatch Errors](#1-shape-mismatch-errors)
2. [Device Mismatch Errors](#2-device-mismatch-errors)
3. [In-Place Operation Errors](#3-in-place-operation-errors)
4. [NaN/Inf in Loss](#4-naninf-in-loss)
5. [Memory Errors](#5-memory-errors)
6. [Common Gotchas](#6-common-gotchas)
7. [torch.autograd.detect_anomaly()](#7-torchautograddetect_anomaly)
8. [Exercises](#8-exercises)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## 1. Shape Mismatch Errors

Shape mismatches are the most common PyTorch errors. They occur when tensors have incompatible shapes for an operation.

**Diagnosis strategy:**
- Print the shapes of all tensors involved
- Check matrix multiplication rules: `(m, n) @ (n, p) -> (m, p)`
- Verify loss function input/target shapes

In [ ]:
# Error 1: Wrong input dimensions to nn.Linear
torch.manual_seed(42)

model = nn.Linear(10, 5)  # expects input with 10 features
x_wrong = torch.randn(32, 20)  # but input has 20 features

print("--- BUG: wrong input features ---")
try:
    out = model(x_wrong)
except RuntimeError as e:
    print(f"Error: {e}")

# FIX: match input features to model's in_features
print("\n--- FIX ---")
x_correct = torch.randn(32, 10)  # 10 features, matching the model
out = model(x_correct)
print(f"Input shape: {x_correct.shape} -> Output shape: {out.shape}")

In [ ]:
# Error 2: CrossEntropyLoss shape mismatch
torch.manual_seed(42)

logits = torch.randn(32, 5)         # (batch, num_classes)
targets_wrong = torch.randint(0, 5, (32, 1))  # wrong: (32, 1) instead of (32,)

loss_fn = nn.CrossEntropyLoss()

print("--- BUG: targets have wrong shape ---")
print(f"Logits shape:  {logits.shape}")
print(f"Targets shape: {targets_wrong.shape} (should be (32,) not (32,1))")
try:
    loss = loss_fn(logits, targets_wrong)
except RuntimeError as e:
    print(f"Error: {e}")

# FIX: squeeze the targets
print("\n--- FIX ---")
targets_correct = targets_wrong.squeeze(1)  # (32,)
loss = loss_fn(logits, targets_correct)
print(f"Targets shape: {targets_correct.shape}")
print(f"Loss: {loss.item():.4f}")

In [ ]:
# Error 3: BCEWithLogitsLoss shape mismatch
torch.manual_seed(42)

logits_bin = torch.randn(32, 1)    # (batch, 1)
targets_bin = torch.randint(0, 2, (32,)).float()  # (32,) -- different shape!

bce_loss_fn = nn.BCEWithLogitsLoss()

print("--- BUG: logits and targets have different shapes ---")
print(f"Logits shape:  {logits_bin.shape}")
print(f"Targets shape: {targets_bin.shape}")

# This may silently broadcast and give wrong results!
loss_wrong = bce_loss_fn(logits_bin, targets_bin)  # broadcasts (32,1) with (32,)
print(f"Loss (silent broadcasting -- may be wrong): {loss_wrong.item():.4f}")

# FIX: make shapes match
print("\n--- FIX ---")
logits_fixed = logits_bin.squeeze(1)  # (32,) to match targets
loss_fixed = bce_loss_fn(logits_fixed, targets_bin)
print(f"Logits shape:  {logits_fixed.shape}")
print(f"Targets shape: {targets_bin.shape}")
print(f"Loss: {loss_fixed.item():.4f}")

In [ ]:
# Error 4: Matrix multiplication shape mismatch
torch.manual_seed(42)

A = torch.randn(3, 4)  # (3, 4)
B = torch.randn(5, 2)  # (5, 2) -- inner dims don't match!

print("--- BUG: matmul inner dimensions mismatch ---")
print(f"A shape: {A.shape}, B shape: {B.shape}")
print(f"Need A.shape[1] == B.shape[0], but {A.shape[1]} != {B.shape[0]}")
try:
    C = A @ B
except RuntimeError as e:
    print(f"Error: {e}")

# FIX: ensure inner dimensions match
print("\n--- FIX ---")
B_fixed = torch.randn(4, 2)  # (4, 2) -- now inner dims match
C = A @ B_fixed
print(f"A shape: {A.shape}, B shape: {B_fixed.shape} -> C shape: {C.shape}")

## 2. Device Mismatch Errors

All tensors in an operation must be on the **same device** (CPU or a specific GPU).

**Error message:** `RuntimeError: expected all tensors to be on the same device`

In [ ]:
# Error: model on GPU, data on CPU (or vice versa)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    # Reproduce the error
    model_gpu = nn.Linear(10, 5).to('cuda')
    x_cpu = torch.randn(4, 10)  # still on CPU!

    print("--- BUG: model on GPU, input on CPU ---")
    print(f"Model device: cuda")
    print(f"Input device: {x_cpu.device}")
    try:
        out = model_gpu(x_cpu)
    except RuntimeError as e:
        print(f"Error: {e}")

    # FIX: move data to the same device
    print("\n--- FIX ---")
    x_gpu = x_cpu.to('cuda')
    out = model_gpu(x_gpu)
    print(f"Input device: {x_gpu.device}")
    print(f"Output: {out.shape} on {out.device}")
else:
    print("No GPU available. Demonstrating the concept on CPU.")
    print("\nThe fix is always the same: move all tensors to the same device.")
    print("  x = x.to(device)")
    print("  model = model.to(device)")

In [ ]:
# Best practice: device-agnostic code pattern
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(42)
model = nn.Linear(10, 5).to(device)
x = torch.randn(4, 10, device=device)   # create directly on device
y = torch.randint(0, 5, (4,), device=device)

out = model(x)
print(f"All on {device}: model={next(model.parameters()).device}, "
      f"x={x.device}, y={y.device}")

## 3. In-Place Operation Errors

In-place operations (those ending with `_`, like `add_`, `relu_`, `+=`) modify a tensor's data directly. This can break autograd's computation graph.

**Error message:** `RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation`

In [ ]:
# Error: in-place operation on a tensor used for gradient computation
torch.manual_seed(42)

print("--- BUG: in-place modification breaks autograd ---")
x = torch.randn(3, requires_grad=True)
y = x * 2  # y depends on x

# In-place operation on y
try:
    y += 1  # equivalent to y.__iadd__(1) -- in-place!
except RuntimeError as e:
    print(f"Error: {e}")

In [ ]:
# FIX: use out-of-place operations
print("--- FIX: out-of-place operation ---")
x = torch.randn(3, requires_grad=True)
y = x * 2
y = y + 1  # creates a NEW tensor -- safe!

loss = y.sum()
loss.backward()
print(f"x.grad = {x.grad}")
print("No error with out-of-place operation.")

In [ ]:
# Common in-place pitfall in model forward pass
torch.manual_seed(42)

class BuggyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 5)

    def forward(self, x):
        x = self.fc(x)
        x.relu_()  # in-place ReLU -- may cause issues!
        return x

class FixedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(5, 5)

    def forward(self, x):
        x = self.fc(x)
        x = torch.relu(x)  # out-of-place ReLU -- safe
        return x

print("Tip: avoid .relu_(), .sigmoid_(), .tanh_() in forward().")
print("Use torch.relu(), torch.sigmoid(), etc. instead.")

## 4. NaN/Inf in Loss

NaN (Not a Number) or Inf values in the loss are a sign of numerical instability.

**Common causes:**

| Cause | Example | Fix |
|---|---|---|
| `log(0)` | `torch.log(probability)` where prob=0 | Use `torch.clamp(p, min=1e-7)` or use built-in loss functions |
| Exploding gradients | Very deep networks, high lr | Gradient clipping, lower lr |
| Division by zero | Custom normalization | Add epsilon: `x / (std + 1e-8)` |
| Overflow in exp | Large logits in softmax | Use `log_softmax` instead of `softmax` + `log` |

In [ ]:
# Error: log(0) produces -inf, which cascades to NaN
print("--- BUG: log(0) ---")

probs = torch.tensor([0.9, 0.1, 0.0])  # one probability is exactly 0
log_probs = torch.log(probs)
print(f"probs:     {probs}")
print(f"log(probs): {log_probs}")
print(f"Contains -inf: {torch.isinf(log_probs).any()}")

# FIX: clamp before log
print("\n--- FIX: clamp ---")
eps = 1e-7
probs_safe = torch.clamp(probs, min=eps)
log_probs_safe = torch.log(probs_safe)
print(f"clamped probs: {probs_safe}")
print(f"log(clamped):  {log_probs_safe}")
print(f"No -inf: {not torch.isinf(log_probs_safe).any()}")

In [ ]:
# Error: exploding gradients with high learning rate
print("--- BUG: exploding gradients (lr too high) ---")
torch.manual_seed(42)

model_explode = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 3),
)

optimizer_explode = optim.SGD(model_explode.parameters(), lr=100.0)  # way too high!
loss_fn = nn.CrossEntropyLoss()
X = torch.randn(32, 10)
y = torch.randint(0, 3, (32,))

for step in range(5):
    logits = model_explode(X)
    loss = loss_fn(logits, y)
    optimizer_explode.zero_grad()
    loss.backward()
    optimizer_explode.step()
    print(f"Step {step}: loss = {loss.item():.6f}")
    if torch.isnan(loss) or torch.isinf(loss):
        print("  --> NaN/Inf detected! Training is broken.")
        break

In [ ]:
# FIX 1: Use a reasonable learning rate
print("--- FIX 1: lower learning rate ---")
torch.manual_seed(42)

model_fix1 = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 3),
)
optimizer_fix1 = optim.SGD(model_fix1.parameters(), lr=0.01)  # reasonable lr

for step in range(5):
    logits = model_fix1(X)
    loss = loss_fn(logits, y)
    optimizer_fix1.zero_grad()
    loss.backward()
    optimizer_fix1.step()
    print(f"Step {step}: loss = {loss.item():.4f}")

In [ ]:
# FIX 2: Gradient clipping
print("--- FIX 2: gradient clipping ---")
torch.manual_seed(42)

model_clip = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 3),
)
optimizer_clip = optim.SGD(model_clip.parameters(), lr=1.0)  # still high, but clipped

max_grad_norm = 1.0

for step in range(5):
    logits = model_clip(X)
    loss = loss_fn(logits, y)
    optimizer_clip.zero_grad()
    loss.backward()

    # Clip gradients
    grad_norm = torch.nn.utils.clip_grad_norm_(model_clip.parameters(), max_grad_norm)

    optimizer_clip.step()
    print(f"Step {step}: loss = {loss.item():.4f}, grad_norm = {grad_norm:.4f}")

## 5. Memory Errors

**Error message:** `CUDA out of memory` or system swap thrashing on CPU.

**Solutions:**
- Reduce batch size
- Use gradient accumulation to simulate larger batches
- Use mixed precision training (Notebook 06)
- Free unused tensors with `del tensor` and `torch.cuda.empty_cache()`

In [ ]:
# Gradient accumulation: simulate batch_size=128 with actual batch_size=32
print("--- Gradient Accumulation ---")
torch.manual_seed(42)

model_accum = nn.Sequential(
    nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 3),
)
optimizer_accum = optim.Adam(model_accum.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# Full dataset
X_full = torch.randn(128, 10)
y_full = torch.randint(0, 3, (128,))

# Instead of one batch of 128, we use 4 micro-batches of 32
accumulation_steps = 4
micro_batch_size = 32

optimizer_accum.zero_grad()

for i in range(accumulation_steps):
    start = i * micro_batch_size
    end = start + micro_batch_size
    X_mb = X_full[start:end]
    y_mb = y_full[start:end]

    logits = model_accum(X_mb)
    loss = loss_fn(logits, y_mb) / accumulation_steps  # scale loss!
    loss.backward()  # gradients accumulate across micro-batches

    print(f"  Micro-batch {i}: loss = {loss.item() * accumulation_steps:.4f}")

# One optimizer step after all micro-batches
optimizer_accum.step()
print("\nOne optimizer step taken (equivalent to batch_size=128).")

In [ ]:
# Memory leak: storing computation graph history
print("--- BUG: storing tensors with grad history ---")
torch.manual_seed(42)

model_leak = nn.Linear(10, 3)
optimizer_leak = optim.Adam(model_leak.parameters())
loss_fn = nn.CrossEntropyLoss()

# BAD: storing loss tensor (retains computation graph!)
losses_bad = []
for step in range(5):
    x = torch.randn(32, 10)
    y = torch.randint(0, 3, (32,))
    logits = model_leak(x)
    loss = loss_fn(logits, y)
    losses_bad.append(loss)  # BAD: stores entire graph!
    optimizer_leak.zero_grad()
    loss.backward()
    optimizer_leak.step()

print(f"Bad: losses_bad[0].grad_fn = {losses_bad[0].grad_fn}")
print("  Graph is retained in memory!\n")

# GOOD: store only the scalar value
losses_good = []
for step in range(5):
    x = torch.randn(32, 10)
    y = torch.randint(0, 3, (32,))
    logits = model_leak(x)
    loss = loss_fn(logits, y)
    losses_good.append(loss.item())  # GOOD: stores just a float
    optimizer_leak.zero_grad()
    loss.backward()
    optimizer_leak.step()

print(f"Good: losses_good = {losses_good}")
print("  Just floats, no graph retained.")

## 6. Common Gotchas

Three things that are easy to forget and hard to debug:

1. Forgetting `model.eval()` during validation
2. Forgetting `torch.no_grad()` during inference
3. Forgetting `optimizer.zero_grad()` in the training loop

In [ ]:
# Gotcha 1: Forgetting model.eval()
print("--- GOTCHA 1: forgetting model.eval() ---")
torch.manual_seed(42)

model_drop = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Dropout(0.5),  # drops 50% of activations during training
    nn.Linear(32, 3),
)

x_eval = torch.randn(1, 10)

# BUG: forget to call model.eval()
model_drop.train()  # still in training mode!
results_train = [model_drop(x_eval).detach() for _ in range(5)]
print("In train mode (dropout active):")
for i, r in enumerate(results_train):
    print(f"  Run {i}: {r}")
print("  -> Different results each time (dropout is random)!\n")

# FIX: call model.eval()
model_drop.eval()
results_eval = [model_drop(x_eval).detach() for _ in range(5)]
print("In eval mode (dropout disabled):")
for i, r in enumerate(results_eval):
    print(f"  Run {i}: {r}")
print("  -> Consistent results!")

In [ ]:
# Gotcha 2: Forgetting torch.no_grad() during inference
print("--- GOTCHA 2: forgetting torch.no_grad() ---")
torch.manual_seed(42)

model_infer = nn.Linear(10, 5)
x = torch.randn(100, 10)

# BUG: without no_grad, PyTorch builds computation graph (wastes memory)
model_infer.eval()
out_with_grad = model_infer(x)
print(f"Without no_grad: output.requires_grad = {out_with_grad.requires_grad}")
print("  -> Computation graph is being tracked unnecessarily!\n")

# FIX: use torch.no_grad()
with torch.no_grad():
    out_no_grad = model_infer(x)
print(f"With no_grad: output.requires_grad = {out_no_grad.requires_grad}")
print("  -> No graph tracked. Faster and uses less memory.")

In [ ]:
# Gotcha 3: Forgetting zero_grad()
print("--- GOTCHA 3: forgetting zero_grad() ---")
torch.manual_seed(42)

model_zg = nn.Linear(5, 1, bias=False)
optimizer_zg = optim.SGD(model_zg.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.randn(4, 5)
y = torch.randn(4, 1)

# BUG: no zero_grad() -- gradients accumulate
print("Without zero_grad (gradients accumulate):")
for step in range(3):
    out = model_zg(x)
    loss = loss_fn(out, y)
    loss.backward()
    grad_norm = model_zg.weight.grad.norm().item()
    print(f"  Step {step}: grad norm = {grad_norm:.4f} (growing!)")
    optimizer_zg.step()

# FIX: zero_grad before backward
print("\nWith zero_grad (correct):")
torch.manual_seed(42)
model_zg2 = nn.Linear(5, 1, bias=False)
optimizer_zg2 = optim.SGD(model_zg2.parameters(), lr=0.1)

for step in range(3):
    optimizer_zg2.zero_grad()  # reset gradients!
    out = model_zg2(x)
    loss = loss_fn(out, y)
    loss.backward()
    grad_norm = model_zg2.weight.grad.norm().item()
    print(f"  Step {step}: grad norm = {grad_norm:.4f} (consistent)")
    optimizer_zg2.step()

## 7. torch.autograd.detect_anomaly()

`torch.autograd.detect_anomaly()` is a context manager that enables anomaly detection in autograd. When a NaN or error occurs during `backward()`, it will provide a **traceback to the forward operation** that caused the issue.

**Trade-off:** It slows down training significantly. Use it only for debugging.

In [ ]:
# Using detect_anomaly to find the source of NaN
print("--- detect_anomaly() demo ---")
torch.manual_seed(42)

# A model that produces NaN via log of negative values
x = torch.randn(5, requires_grad=True)

try:
    with torch.autograd.detect_anomaly():
        y = x * 2
        z = torch.log(y)  # log of negative values -> NaN
        loss = z.sum()
        loss.backward()   # detect_anomaly catches the NaN here
except RuntimeError as e:
    print(f"Anomaly detected: {str(e)[:200]}...")
    print("\ndetect_anomaly() told us WHERE the NaN originated.")
    print("Fix: ensure inputs to log() are positive.")

In [ ]:
# Alternative: set it globally for a debugging session
# torch.autograd.set_detect_anomaly(True)  # enable globally
# ... training code ...
# torch.autograd.set_detect_anomaly(False)  # disable when done

print("Usage patterns:")
print("  1. Context manager: with torch.autograd.detect_anomaly(): ...")
print("  2. Global toggle:   torch.autograd.set_detect_anomaly(True)")
print("\nRemember to disable it after debugging -- it slows training.")

## 8. Exercises

### Exercise: Fix the Buggy Training Code

The code below has **5 bugs**. Find and fix all of them. The bugs are among the errors covered in this notebook.

**Hints:**
- Each bug is a common mistake from sections 1-6
- The code should train without errors and achieve reasonable accuracy
- Look carefully at shapes, devices, modes, gradients, and loss functions

In [ ]:
# =============================================
# BUGGY CODE -- Find and fix 5 bugs!
# =============================================

# torch.manual_seed(42)
#
# # Data
# X = torch.randn(200, 10)
# y = torch.randint(0, 3, (200, 1))   # BUG 1: wrong shape for CrossEntropyLoss
#
# # Model
# model = nn.Sequential(
#     nn.Linear(10, 32),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(32, 3),
#     nn.Softmax(dim=1),               # BUG 2: double softmax with CrossEntropyLoss
# )
#
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.01)
#
# # Training
# losses = []
# for epoch in range(50):
#     # BUG 3: missing optimizer.zero_grad()
#     logits = model(X)
#     loss = loss_fn(logits, y)
#     loss.backward()
#     optimizer.step()
#     losses.append(loss)              # BUG 4: storing tensor instead of .item()
#
# # Evaluation
# # BUG 5: missing model.eval() and torch.no_grad()
# logits = model(X)
# preds = logits.argmax(dim=1)
# acc = (preds == y.squeeze()).float().mean()
# print(f"Accuracy: {acc.item():.4f}")

### Solution

In [ ]:
# =============================================
# FIXED CODE
# =============================================

torch.manual_seed(42)

# Data
X = torch.randn(200, 10)
y = torch.randint(0, 3, (200,))              # FIX 1: shape (200,) not (200, 1)

# Model
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(32, 3),
    # FIX 2: removed nn.Softmax -- CrossEntropyLoss applies it internally
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training
losses = []
for epoch in range(50):
    model.train()
    optimizer.zero_grad()                      # FIX 3: zero gradients
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())                 # FIX 4: .item() to store float

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: loss = {losses[-1]:.4f}")

# Evaluation
model.eval()                                   # FIX 5a: switch to eval mode
with torch.no_grad():                          # FIX 5b: disable grad tracking
    logits = model(X)
    preds = logits.argmax(dim=1)
    acc = (preds == y).float().mean()
    print(f"\nFinal Accuracy: {acc.item():.4f}")

print("\nAll 5 bugs fixed!")

## 9. Common Mistakes & Debugging Tips

### Quick Reference Checklist

When your training loop is not working, check these in order:

1. **Shapes**: Print tensor shapes at every step. Most bugs are shape bugs.
   ```python
   print(f"input: {x.shape}, output: {out.shape}, target: {y.shape}")
   ```

2. **Devices**: Ensure model and all data are on the same device.
   ```python
   print(f"model: {next(model.parameters()).device}, x: {x.device}")
   ```

3. **Loss function inputs**:
   - `CrossEntropyLoss`: logits `(N, C)` + targets `(N,)` as `torch.long`
   - `BCEWithLogitsLoss`: logits `(N,)` + targets `(N,)` as `torch.float`
   - Never apply `softmax`/`sigmoid` before these loss functions

4. **Gradient flow**:
   - Call `optimizer.zero_grad()` before `loss.backward()`
   - Avoid in-place operations on tensors that require gradients
   - Use `loss.item()` when storing loss values for logging

5. **Training vs eval mode**:
   - `model.train()` before training
   - `model.eval()` + `torch.no_grad()` before validation/inference

6. **NaN/Inf debugging**:
   - Use `torch.autograd.detect_anomaly()` to trace the source
   - Check for `log(0)`, division by zero, very high learning rates
   - Apply gradient clipping: `torch.nn.utils.clip_grad_norm_`

7. **Memory issues**:
   - Reduce batch size
   - Use gradient accumulation
   - Use `.item()` or `.detach()` to avoid retaining computation graphs
   - Call `torch.cuda.empty_cache()` to free cached GPU memory